# Notebook 05 — Your First Simple RAG Application

**Use case:** Ask questions from Meridian policy documents.

This is a teaching notebook. It deliberately uses the minimum moving parts:

1. **Load** PDF, TXT, and Markdown documents.
2. **Chunk** the documents into smaller passages.
3. **Embed** the passages and keep the vectors in memory.
4. **Retrieve** the most relevant passages for a question.
5. **Generate** a grounded answer from those passages.

There is **no SQL, no Delta table, no managed vector database, and no agent framework** in this first lesson. The in-memory NumPy matrix is our simple vector store.


## The RAG idea

A language model does not automatically know your private documents. RAG gives the model relevant document passages before it answers.

```text
Documents → Chunks → Embeddings → Similarity Search → Context → LLM Answer
```

- An **embedding** is a list of numbers representing the meaning of some text.
- Similar meanings produce vectors that are close together.
- We compare the question vector with every chunk vector and retrieve the closest chunks.


## 0. Install the PDF reader

Databricks already provides NumPy and MLflow. We only install `pypdf` for extracting text from PDF files.


In [0]:
%pip install -q pypdf


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()


## 1. Configuration

The Databricks interface shows your project under **Workspace → Nuclear_Enterprise_360**. Depending on how the workspace exposes personal files to Python, the physical path can include `/Users/<email>`.

The code checks both possible paths automatically and uses the one that exists.


In [0]:
import os
from pathlib import Path

import numpy as np
from mlflow.deployments import get_deploy_client
from pypdf import PdfReader

# Document folder path - change this according to where you are saving thr filee
DOCUMENT_FOLDER = "/Workspace/Nuclear_Enterprise_360/meridian_leave_policies"

if not os.path.isdir(DOCUMENT_FOLDER):
    raise FileNotFoundError(
        f"The folder was not found at {DOCUMENT_FOLDER}. "
        "Please verify the Databricks workspace path."
    )

EMBEDDING_MODEL = "databricks-qwen3-embedding-0-6b"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"

print("Reading documents from:", DOCUMENT_FOLDER)


Reading documents from: /Workspace/Nuclear_Enterprise_360/meridian_leave_policies


## 2. Load PDF, TXT, and Markdown files

Each extracted text block keeps its filename and PDF page number. This allows us to show the source used for an answer.

Scanned image-only PDFs need OCR and are outside this beginner notebook.


In [0]:
def read_document(path):
    """Return text blocks from one PDF, TXT, or Markdown file."""
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        reader = PdfReader(str(path))
        return [
            {"source": path.name, "page": page_number, "text": page.extract_text() or ""}
            for page_number, page in enumerate(reader.pages, start=1)
        ]

    # TXT and Markdown are normal UTF-8 text files.
    text = path.read_text(encoding="utf-8", errors="ignore")
    return [{"source": path.name, "page": None, "text": text}]


files = sorted(
    path
    for path in Path(DOCUMENT_FOLDER).rglob("*")
    if path.is_file() and path.suffix.lower() in {".pdf", ".txt", ".md"}
)

if not files:
    raise ValueError("No PDF, TXT, or Markdown files were found.")

blocks = []
for file in files:
    blocks.extend(read_document(file))

blocks = [block for block in blocks if block["text"].strip()]

print(f"Loaded {len(files)} files and {len(blocks)} text blocks.")
for file in files:
    print("-", file.name)


Loaded 1 files and 9 text blocks.
- leave_absence_policy_v3.1.pdf


## 3. Split the documents into chunks

We cannot send every document to the language model for every question. We split the documents into smaller overlapping passages.

The overlap helps preserve meaning when an important sentence crosses a chunk boundary.


In [0]:
def split_text(text, chunk_size=1200, overlap=200):
    text = " ".join(text.split())
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])

        if end == len(text):
            break

        start = end - overlap

    return chunks


chunks = []
for block in blocks:
    for chunk_number, text in enumerate(split_text(block["text"]), start=1):
        chunks.append({
            "source": block["source"],
            "page": block["page"],
            "chunk": chunk_number,
            "text": text,
        })

print(f"Created {len(chunks)} chunks.")
print("\nExample chunk:\n", chunks[0]["text"][:700])


Created 15 chunks.

Example chunk:
 MERIDIAN GROUP DUBAI · ABU DHABI · RIYADH · CAIRO · BENGALURU This document is the property of Meridian Group. Internal distribution only. POL-HR-002 · Version 3.1 · Effective 2026-01-01 LEAVE Leave and Absence Policy Annual, sick, parental and special leave entitlements across all Meridian Group entities. Document code POL-HR-002 Version 3.1 Effective date 2026-01-01 Next review date 2027-01-01 Document owner Head of Human Resources Applies to All employees, all locations Classification Internal


## 4. Create embeddings and the in-memory vector store

We send every chunk to a Databricks embedding model. The returned vectors are stored in a NumPy matrix called `vector_store`.

This is intentionally simple and transparent. In a production system, the same concept can be scaled with Databricks AI Search or another managed vector database.


In [0]:
import time

client = get_deploy_client("databricks")


def create_embeddings(texts, batch_size=32):
    """Create embeddings in small batches with retry logic."""
    all_vectors = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        
        # Retry with exponential backoff on rate limit errors
        max_retries = 5
        for attempt in range(max_retries):
            try:
                response = client.predict(
                    endpoint=EMBEDDING_MODEL,
                    inputs={"input": batch},
                )
                ordered = sorted(response["data"], key=lambda item: item["index"])
                all_vectors.extend(item["embedding"] for item in ordered)
                break  # Success, exit retry loop
            except Exception as e:
                if "429" in str(e) or "REQUEST_LIMIT_EXCEEDED" in str(e):
                    if attempt < max_retries - 1:
                        wait_time = 2 ** attempt  # Exponential backoff: 1, 2, 4, 8, 16 seconds
                        print(f"Rate limit hit. Waiting {wait_time}s before retry {attempt + 1}/{max_retries}...")
                        time.sleep(wait_time)
                    else:
                        raise  # Max retries exceeded
                else:
                    raise  # Different error, propagate immediately

    return np.array(all_vectors, dtype="float32")


# This matrix is our simple vector store: one vector for every document chunk.
vector_store = create_embeddings([chunk["text"] for chunk in chunks])

# Normalize once so cosine similarity becomes a simple dot product.
vector_store = vector_store / np.linalg.norm(vector_store, axis=1, keepdims=True)

print("Vector store shape:", vector_store.shape)
print("Rows = chunks; columns = embedding dimensions")


Vector store shape: (15, 1024)
Rows = chunks; columns = embedding dimensions


## 5. Retrieve the most relevant chunks

The question is embedded using the same model. We calculate cosine similarity between the question vector and every chunk vector, then return the best matches.


In [0]:
def retrieve(question, top_k=3):
    question_vector = create_embeddings([question])[0]
    question_vector = question_vector / np.linalg.norm(question_vector)

    scores = vector_store @ question_vector
    best_indexes = np.argsort(scores)[::-1][:top_k]

    results = []
    for index in best_indexes:
        result = chunks[index].copy()
        result["score"] = float(scores[index])
        results.append(result)

    return results


# Test retrieval before asking the language model.
test_results = retrieve("How much annual leaves are employees entitled?")

for result in test_results:
    page = f", page {result['page']}" if result["page"] else ""
    print(f"{result['score']:.3f} | {result['source']}{page}")


0.614 | leave_absence_policy_v3.1.pdf, page 3
0.572 | leave_absence_policy_v3.1.pdf, page 5
0.570 | leave_absence_policy_v3.1.pdf, page 3


## 6. Generate a grounded answer

The `ask()` function performs the two steps that happen at question time:

1. Retrieve the best document chunks.
2. Give those chunks to the chat model as context.

The prompt tells the model to answer only from the retrieved context and cite the source numbers.


In [0]:
def ask(question, top_k=3):
    retrieved = retrieve(question, top_k)

    context_parts = []
    for number, item in enumerate(retrieved, start=1):
        page = f", page {item['page']}" if item["page"] else ""
        context_parts.append(
            f"SOURCE [{number}] — {item['source']}{page}\n{item['text']}"
        )

    context = "\n\n".join(context_parts)

    messages = [
        {
            "role": "system",
            "content": (
                "Answer only from the supplied document context. "
                "Cite factual statements with [1], [2], and so on. "
                "If the answer is not in the context, say: "
                "I could not find this in the provided documents."
            ),
        },
        {
            "role": "user",
            "content": f"QUESTION:\n{question}\n\nDOCUMENT CONTEXT:\n{context}",
        },
    ]

    response = client.predict(
        endpoint=CHAT_MODEL,
        inputs={"messages": messages, "temperature": 0.1, "max_tokens": 700},
    )
    answer = response["choices"][0]["message"]["content"]

    print("QUESTION\n", question)
    print("\nANSWER\n", answer)
    print("\nRETRIEVED SOURCES")

    for number, item in enumerate(retrieved, start=1):
        page = f", page {item['page']}" if item["page"] else ""
        print(f"[{number}] {item['source']}{page} | score={item['score']:.3f}")

    return answer


## 7. Ask your first question

Change the question below and rerun only this cell. The documents and vectors do not need to be rebuilt for every question.


In [0]:
ask("What approvals are required?")


QUESTION
 What approvals are required?

ANSWER
 According to the document context, the approvals required for a leave request are as follows:

1. The line manager's approval [1]: The line manager reviews the request and either approves or rejects it.
2. Department head's approval (in case of escalation) [1]: If the line manager rejects the request, the employee may escalate to the department head.
3. Human Resources' approval (in case of further escalation) [1]: If the department head rejects the request, the employee may further escalate to Human Resources.

Note that Finance and Customer Success may also declare blackout periods, which may affect the approval of leave requests [3]. However, this is not a direct approval requirement, but rather a consideration that managers must take into account when reviewing leave requests.

RETRIEVED SOURCES
[1] leave_absence_policy_v3.1.pdf, page 8 | score=0.374
[2] leave_absence_policy_v3.1.pdf, page 5 | score=0.347
[3] leave_absence_policy_v3.1

"According to the document context, the approvals required for a leave request are as follows:\n\n1. The line manager's approval [1]: The line manager reviews the request and either approves or rejects it.\n2. Department head's approval (in case of escalation) [1]: If the line manager rejects the request, the employee may escalate to the department head.\n3. Human Resources' approval (in case of further escalation) [1]: If the department head rejects the request, the employee may further escalate to Human Resources.\n\nNote that Finance and Customer Success may also declare blackout periods, which may affect the approval of leave requests [3]. However, this is not a direct approval requirement, but rather a consideration that managers must take into account when reviewing leave requests."

## Questions students can try

```python
ask("What approvals are required?")
ask("Who is responsible for reviewing compliance?")
ask("What escalation steps are described?")
ask("Summarize the policy controls in five points.")
ask("What does the policy say about something that is not mentioned?")
```

## Teaching summary

| Stage | What happens in the code |
|---|---|
| Load | Read PDF, TXT, and Markdown files |
| Chunk | Split long text into overlapping passages |
| Embed | Convert every passage into a numeric vector |
| Store | Keep all vectors in the NumPy `vector_store` matrix |
| Retrieve | Find chunks closest to the question vector |
| Generate | Send only those chunks to the language model |

This is the minimum complete RAG pattern. After students understand this notebook, you can introduce persistent vector stores, metadata filtering, SQL data, evaluation, governance, and agents as separate upgrades.


In [0]:
%pip install -q gradio 'fastapi>=0.100.0'

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import gradio as gr

print("[DEBUG] Gradio interface loaded successfully")

def chat_with_history(message, history):
    """
    Chatbot function that maintains conversation history.
    
    Args:
        message: The user's current question
        history: List of [user_msg, bot_msg] pairs from previous turns
    
    Returns:
        The assistant's response
    """
    # Use the existing ask() function to get the answer
    print(f"[DEBUG] chat_with_history called with message: {message}")
    print(f"[DEBUG] History length: {len(history)}")
    
    retrieved = retrieve(message, top_k=3)
    print(f"[DEBUG] Retrieved {len(retrieved)} chunks")

    context_parts = []
    for number, item in enumerate(retrieved, start=1):
        page = f", page {item['page']}" if item["page"] else ""
        context_parts.append(
            f"SOURCE [{number}] — {item['source']}{page}\n{item['text']}"
        )

    context = "\n\n".join(context_parts)
    print(f"[DEBUG] Context assembled, length: {len(context)} characters")

    # Build conversation context from history
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant answering questions about Meridian leave policies. "
                "Answer only from the supplied document context. "
                "Cite factual statements with [1], [2], and so on. "
                "If the answer is not in the context, say: "
                "I could not find this in the provided documents."
            ),
        },
    ]
    
    # Add conversation history
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    
    # Add current question with context
    messages.append({
        "role": "user",
        "content": f"QUESTION:\n{message}\n\nDOCUMENT CONTEXT:\n{context}",
    })
    
    print(f"[DEBUG] Total messages in conversation: {len(messages)}")

    try:
        print(f"[DEBUG] Calling chat model: {CHAT_MODEL}")
        response = client.predict(
            endpoint=CHAT_MODEL,
            inputs={"messages": messages, "temperature": 0.1, "max_tokens": 700},
        )
        answer = response["choices"][0]["message"]["content"]
        print(f"[DEBUG] Received answer, length: {len(answer)} characters")
    except Exception as e:
        print(f"[DEBUG ERROR] Exception during model prediction: {str(e)}")
        raise
    
    # Add source citations at the end
    sources = "\n\n---\n**Retrieved Sources:**\n"
    for number, item in enumerate(retrieved, start=1):
        page = f", page {item['page']}" if item["page"] else ""
        sources += f"\n[{number}] {item['source']}{page} (score={item['score']:.3f})"
    
    result = answer + sources
    print(f"[DEBUG] Returning response with sources")
    return result


# Create the Gradio interface
demo = gr.ChatInterface(
    fn=chat_with_history,
    title="🏢 Meridian HR Policy Assistant",
    description="Ask questions about Meridian leave policies. The assistant will answer based on the policy documents and cite its sources.",
    examples=[
        "What approvals are required?",
        "Who is responsible for reviewing compliance?",
        "What are the most important requirements and responsibilities?",
        "What escalation steps are described?",
        "Summarize the policy controls in five points.",
    ],
)

# Launch the app
print("[DEBUG] Launching Gradio interface...")
try:
    demo.launch(share=True, height=800)
    print("[DEBUG] Gradio interface launched successfully")
except Exception as e:
    print(f"[DEBUG ERROR] Failed to launch Gradio: {str(e)}")
    raise

[DEBUG] Gradio interface loaded successfully
[DEBUG] Launching Gradio interface...
* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://c5aa43f1ccf18f14cd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[DEBUG] Gradio interface launched successfully
